# Laboratório — Pré-processamento, pipelines e data leakage

Este notebook acompanha a **Aula 03 do módulo 03 · Machine Learning clássico**.

**Pergunta:** como preservar a fronteira entre desenvolvimento e avaliação quando imputação, escala, encoding e seleção de atributos também aprendem com dados?

Ao final, você terá:

- um pipeline tabular misto ajustado somente no treino;
- validação cruzada reproduzível e teste final isolado;
- inspeção do estado aprendido por imputador, scaler e encoder;
- uma contraprova em que seleção global de features fabrica desempenho em dados sem sinal.

> Os dados são sintéticos e servem para demonstrar o método. Eles não medem um produto real.

## 1. Ambiente e dependências

Requisitos mínimos:

- Python 3.11;
- NumPy 2.0;
- pandas 2.0;
- Matplotlib 3.8;
- scikit-learn 1.5.

O notebook não usa rede, credenciais nem arquivos externos. A seed fixa reproduz esta sequência pseudoaleatória em ambiente compatível.

In [ ]:
import platform
from importlib.metadata import version

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    roc_auc_score,
)
from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 20260908
rng = np.random.default_rng(SEED)

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("Matplotlib:", version("matplotlib"))
print("scikit-learn:", sklearn.__version__)
print("Seed:", SEED)

## 2. Protocolo antes dos resultados

### Contrato de predição

- **Decisão apoiada:** priorizar contatos que devem receber acompanhamento humano.
- **Unidade de análise:** cliente no instante da triagem.
- **Target:** necessidade de acompanhamento em até 30 dias.
- **Features permitidas:** idade, renda declarada, meses de relacionamento, região e canal atual.
- **Colunas proibidas:** resultado da revisão, ação humana posterior e qualquer agregação produzida após a triagem.
- **Cenário de generalização:** novos clientes da mesma operação e período próximo.
- **Desenvolvimento:** 75% das unidades; usado em validação cruzada.
- **Teste:** 25%, reservado e aberto uma vez após congelar o pipeline.
- **Métricas didáticas:** ROC AUC, average precision e Brier score.

Hipótese metodológica: o pipeline completo preservará a fronteira de `fit`. Na contraprova sem sinal, selecionar features antes da CV produzirá métrica artificialmente maior que selecionar dentro de cada fold.

## 3. Dados sintéticos mistos

O target é amostrado de uma probabilidade logística que depende de algumas variáveis observáveis. Introduzimos ausências em renda e região **antes** do split. `client_id` identifica a unidade, mas não entra no modelo.

In [ ]:
n = 900
idade = np.clip(rng.normal(41, 12, n), 18, 80)
renda = rng.lognormal(mean=8.55, sigma=0.55, size=n)
meses = rng.integers(1, 121, size=n)
regiao = rng.choice(["Norte", "Nordeste", "Sudeste", "Sul"], size=n,
                    p=[0.18, 0.27, 0.38, 0.17]).astype(object)
canal = rng.choice(["app", "web", "telefone"], size=n,
                   p=[0.48, 0.37, 0.15])

logit = (
    -0.9
    + 0.045 * (idade - 40)
    - 0.55 * (np.log(renda) - 8.55)
    - 0.009 * (meses - 48)
    + 0.55 * (regiao == "Norte")
    + 0.40 * (canal == "telefone")
)
prob = 1 / (1 + np.exp(-logit))
y = rng.binomial(1, prob)

renda[rng.choice(n, 72, replace=False)] = np.nan
regiao[rng.choice(n, 36, replace=False)] = np.nan

data = pd.DataFrame({
    "client_id": [f"C{i:04d}" for i in range(n)],
    "idade": idade,
    "renda": renda,
    "meses_relacionamento": meses,
    "regiao": regiao,
    "canal": canal,
    "necessita_acompanhamento": y,
})

print(data.head(3).to_string(index=False))
print("\nShape:", data.shape)
print("Prevalência:", round(data["necessita_acompanhamento"].mean(), 6))
print("Ausências:", data.isna().sum().to_dict())

### Verificações de entrada

Estas asserções transformam parte do contrato em evidência executável. Em produção, acrescente schema, tipos, intervalos, timestamps e regras de linhagem.

In [ ]:
TARGET = "necessita_acompanhamento"
ID_COL = "client_id"
NUMERIC = ["idade", "renda", "meses_relacionamento"]
CATEGORICAL = ["regiao", "canal"]
FEATURES = NUMERIC + CATEGORICAL
FORBIDDEN = {TARGET, "resultado_revisao", "acao_humana_posterior"}

assert data[ID_COL].is_unique
assert set(data[TARGET].unique()) == {0, 1}
assert FORBIDDEN.isdisjoint(FEATURES)
assert set(FEATURES).issubset(data.columns)
assert data["idade"].between(18, 80).all()
assert (data["meses_relacionamento"] > 0).all()
assert data[NUMERIC].replace([np.inf, -np.inf], np.nan).notna().any().all()

print("Validação de entrada: OK")

## 4. Separe antes de ajustar qualquer estado

O teste é criado agora e permanecerá intocado durante a escolha do pipeline. A estratificação preserva aproximadamente a prevalência das classes; ela não substitui splits por tempo ou entidade quando esses forem exigidos pelo contrato.

In [ ]:
X = data[FEATURES].copy()
y = data[TARGET].copy()
ids = data[ID_COL].copy()

X_dev, X_test, y_dev, y_test, ids_dev, ids_test = train_test_split(
    X, y, ids,
    test_size=0.25,
    stratify=y,
    random_state=SEED,
)

assert set(ids_dev).isdisjoint(set(ids_test))
assert X_dev.columns.tolist() == X_test.columns.tolist() == FEATURES
assert len(X_dev) + len(X_test) == n

print("Desenvolvimento:", X_dev.shape, "prevalência", round(y_dev.mean(), 6))
print("Teste reservado:", X_test.shape, "prevalência", round(y_test.mean(), 6))

## 5. Pipeline por tipo de coluna

O bloco numérico aprende medianas, indicadores de ausência, médias e desvios. O categórico aprende moda e vocabulário. `handle_unknown="ignore"` mantém a inferência operacional quando surge categoria não observada, sem fingir que o modelo aprendeu seu efeito.

In [ ]:
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer([
    ("num", numeric_pipe, NUMERIC),
    ("cat", categorical_pipe, CATEGORICAL),
], remainder="drop")

pipeline = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=2_000, random_state=SEED)),
])

print(pipeline)

## 6. Validação cruzada somente no desenvolvimento

Usamos os mesmos cinco folds para o `DummyClassifier` e o pipeline. O baseline ignora as features. Não tocamos no teste nesta etapa.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = {"roc_auc": "roc_auc", "avg_precision": "average_precision"}

dummy_scores = cross_validate(
    DummyClassifier(strategy="prior"),
    np.zeros((len(y_dev), 1)),
    y_dev,
    cv=cv,
    scoring=scoring,
)
pipeline_scores = cross_validate(
    pipeline,
    X_dev,
    y_dev,
    cv=cv,
    scoring=scoring,
)

cv_summary = pd.DataFrame({
    "modelo": ["Dummy prior", "Pipeline logístico"],
    "roc_auc_media": [dummy_scores["test_roc_auc"].mean(),
                       pipeline_scores["test_roc_auc"].mean()],
    "roc_auc_dp": [dummy_scores["test_roc_auc"].std(ddof=1),
                    pipeline_scores["test_roc_auc"].std(ddof=1)],
    "ap_media": [dummy_scores["test_avg_precision"].mean(),
                  pipeline_scores["test_avg_precision"].mean()],
})

print(cv_summary.round(6).to_string(index=False))
assert pipeline_scores["test_roc_auc"].mean() > 0.60
assert len(pipeline_scores["test_roc_auc"]) == 5

## 7. Congele, ajuste no desenvolvimento e abra o teste uma vez

Agora a configuração está congelada. Ajustamos o pipeline em todo o desenvolvimento e aplicamos somente `transform` e `predict_proba` ao teste.

In [ ]:
pipeline.fit(X_dev, y_dev)
test_probability = pipeline.predict_proba(X_test)[:, 1]

test_metrics = {
    "roc_auc": roc_auc_score(y_test, test_probability),
    "average_precision": average_precision_score(y_test, test_probability),
    "brier": brier_score_loss(y_test, test_probability),
}
print({k: round(v, 6) for k, v in test_metrics.items()})

assert np.isfinite(test_probability).all()
assert ((0 <= test_probability) & (test_probability <= 1)).all()
assert test_metrics["roc_auc"] > 0.60

## 8. Inspecione o estado aprendido

Uma pipeline auditável não é uma caixa fechada. Comparamos as medianas do imputador às medianas de `X_dev` e verificamos que o encoder conhece apenas categorias aprendidas no desenvolvimento.

In [ ]:
fitted_preprocess = pipeline.named_steps["preprocess"]
fitted_num = fitted_preprocess.named_transformers_["num"]
fitted_cat = fitted_preprocess.named_transformers_["cat"]

learned_medians = fitted_num.named_steps["imputer"].statistics_
expected_medians = X_dev[NUMERIC].median().to_numpy()
np.testing.assert_allclose(learned_medians, expected_medians, rtol=0, atol=1e-12)

encoder = fitted_cat.named_steps["onehot"]
feature_names = fitted_preprocess.get_feature_names_out()

print("Medianas aprendidas:", dict(zip(NUMERIC, np.round(learned_medians, 4))))
print("Categorias aprendidas:",
      {col: cats.tolist() for col, cats in zip(CATEGORICAL, encoder.categories_)})
print("Número de features transformadas:", len(feature_names))
print("Primeiros nomes:", feature_names[:8].tolist())

### Categoria desconhecida

Criamos uma linha válida com região `Centro-Oeste`, ausente no conjunto gerado. A transformação deve manter o mesmo shape e não alterar o estado do encoder.

In [ ]:
unseen = pd.DataFrame({
    "idade": [37.0],
    "renda": [5_500.0],
    "meses_relacionamento": [12],
    "regiao": ["Centro-Oeste"],
    "canal": ["app"],
})

categories_before = [cats.copy() for cats in encoder.categories_]
unseen_transformed = fitted_preprocess.transform(unseen)
categories_after = encoder.categories_

assert unseen_transformed.shape[1] == len(feature_names)
for before, after in zip(categories_before, categories_after):
    np.testing.assert_array_equal(before, after)

print("Shape da categoria desconhecida:", unseen_transformed.shape)
print("Estado do encoder permaneceu inalterado: OK")

## 9. Contraprova: seleção global fabrica sinal

Geramos rótulos balanceados e 5.000 features independentes. Por construção, não há relação preditiva. Ainda assim, procurar em todas as features com todos os rótulos encontra associações ocasionais.

- **Versão contaminada:** `SelectKBest.fit_transform` antes da CV.
- **Versão correta:** `SelectKBest` dentro de um pipeline clonado em cada fold.

Os folds e o classificador são os mesmos. A única diferença é a fronteira de `fit` do seletor.

In [ ]:
rng_noise = np.random.default_rng(SEED + 1)
n_noise, p_noise = 200, 5_000
X_noise = rng_noise.normal(size=(n_noise, p_noise))
y_noise = np.array([0, 1] * (n_noise // 2))
rng_noise.shuffle(y_noise)
noise_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

selector_global = SelectKBest(score_func=f_classif, k=20)
X_selected_global = selector_global.fit_transform(X_noise, y_noise)  # leakage intencional
leaky = cross_validate(
    LogisticRegression(max_iter=2_000, C=0.5),
    X_selected_global,
    y_noise,
    cv=noise_cv,
    scoring="roc_auc",
)["test_score"]

correct_selector = Pipeline([
    ("select", SelectKBest(score_func=f_classif, k=20)),
    ("model", LogisticRegression(max_iter=2_000, C=0.5)),
])
correct = cross_validate(
    correct_selector,
    X_noise,
    y_noise,
    cv=noise_cv,
    scoring="roc_auc",
)["test_score"]

leakage_summary = pd.DataFrame({
    "fold": np.arange(1, 6),
    "seleção_global_contaminada": leaky,
    "seleção_dentro_do_pipeline": correct,
})
print(leakage_summary.round(6).to_string(index=False))
print("\nMédias:", leakage_summary.drop(columns="fold").mean().round(6).to_dict())

assert leaky.mean() > correct.mean() + 0.20
assert abs(correct.mean() - 0.5) < 0.16

### Visualização da inflação

O gráfico mostra a distribuição dos cinco folds. Ele não compara dois algoritmos: compara dois **protocolos** para o mesmo problema sem sinal.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

axes[0].bar(cv_summary["modelo"], cv_summary["roc_auc_media"],
            color=["#94a3b8", "#2563eb"])
axes[0].axhline(0.5, color="black", linestyle="--", linewidth=1)
axes[0].set_ylim(0.4, 1.0)
axes[0].set_ylabel("ROC AUC média na CV")
axes[0].set_title("Dados tabulares com sinal")
axes[0].tick_params(axis="x", rotation=12)

axes[1].plot(range(1, 6), leaky, "o-", label="seleção global (leakage)")
axes[1].plot(range(1, 6), correct, "o-", label="seleção no pipeline")
axes[1].axhline(0.5, color="black", linestyle="--", linewidth=1, label="acaso")
axes[1].set_xticks(range(1, 6))
axes[1].set_ylim(0.25, 1.0)
axes[1].set_xlabel("Fold")
axes[1].set_ylabel("ROC AUC")
axes[1].set_title("5.000 features e nenhum sinal")
axes[1].legend(fontsize=8)

fig.suptitle("A fronteira de fit muda a validade da métrica")
fig.tight_layout()
plt.show()

**Texto alternativo:** à esquerda, o pipeline logístico supera o baseline nos dados com sinal. À direita, a seleção global apresenta ROC AUC muito acima do acaso nos dados aleatórios, enquanto a seleção dentro de cada fold oscila perto de 0,5.

## 10. Verificação final e interpretação

As verificações abaixo não provam ausência de qualquer leakage. Elas confirmam propriedades observáveis deste laboratório: separação de IDs, ausência de colunas proibidas, teste aberto somente após congelamento e diferença prevista na contraprova.

In [ ]:
checks = {
    "ids_dev_teste_disjuntos": set(ids_dev).isdisjoint(set(ids_test)),
    "target_fora_das_features": TARGET not in FEATURES,
    "colunas_proibidas_ausentes": FORBIDDEN.isdisjoint(FEATURES),
    "medianas_somente_do_dev": np.allclose(learned_medians, expected_medians),
    "probabilidades_validas": bool(((0 <= test_probability) & (test_probability <= 1)).all()),
    "contraprova_detectou_inflacao": bool(leaky.mean() > correct.mean() + 0.20),
}

assert all(checks.values())
for name, passed in checks.items():
    print(f"{name}: {'OK' if passed else 'FALHOU'}")

print("\nResumo numérico")
print("CV pipeline ROC AUC:", round(pipeline_scores["test_roc_auc"].mean(), 6))
print("Teste final ROC AUC:", round(test_metrics["roc_auc"], 6))
print("Seleção global em ruído:", round(leaky.mean(), 6))
print("Seleção correta em ruído:", round(correct.mean(), 6))

## 11. Takeaways

1. Imputação, escala, encoding e seleção aprendem estado; pertencem ao treinamento.
2. O teste deve ser separado antes de qualquer `fit` e aberto após congelar a configuração.
3. Em CV, o pipeline refaz todas as etapas ajustáveis dentro do treino de cada fold.
4. Seleção global encontrou correlações espúrias em dados sem sinal e inflou a ROC AUC.
5. `handle_unknown="ignore"` evita falha, mas não ensina o efeito de uma categoria inédita.
6. Inspecionar estado, nomes e shapes ajuda a auditar o artefato.
7. Pipeline não corrige futuro, grupos repetidos, proxies do target ou contrato incorreto.
8. Em um sistema real, registre schema, linhagem, timestamps, versões e testes de separação.

### Limite da conclusão

Este notebook demonstra, em dados sintéticos, que mover uma operação supervisionada para fora da fronteira dos folds pode fabricar desempenho. Ele não mede a frequência nem a magnitude do problema em um dataset real específico.

### Próximo passo

Na Aula 04, reutilize esta estrutura para ajustar regressão linear. Mantenha transformações aprendidas dentro do pipeline e analise os resíduos apenas segundo o papel de cada partição.

## 12. Exercício de transferência

Adapte o pipeline a um dataset tabular próprio e entregue:

- contrato de predição com unidade e instante (t_0);
- lista autorizada e lista proibida de features;
- justificativa do splitter;
- pipeline completo;
- baseline;
- pelo menos quatro asserts metodológicos;
- comparação entre uma versão correta e uma contaminação deliberada;
- conclusão que explicite o que os resultados não demonstram.

Se houver grupos ou tempo, substitua o `StratifiedKFold` por uma estratégia coerente com o uso. Não escolha o splitter pela métrica mais alta.